# Oficina Mecânica - Simulação de Fluxo Real
Simulação do atendimento completo de um cliente, desde a entrada do veículo até a emissão da nota fiscal.

## 1. Configuração
Carregamento das dependências e conexão com o banco.

In [1]:
%maven com.j256.ormlite:ormlite-core:6.1
%maven com.j256.ormlite:ormlite-jdbc:6.1
%maven org.xerial:sqlite-jdbc:3.45.1.0
%maven org.slf4j:slf4j-api:2.0.12
%maven ch.qos.logback:logback-classic:1.4.14

In [2]:
%classpath add jar ../target/classes

import br.edu.ufg.oficina.domain.*;
import br.edu.ufg.oficina.repository.*;
import java.util.Date;

System.out.println("Classes importadas com sucesso!");

Classes importadas com sucesso!


In [3]:
Database database = new Database("oficina.db");
database.getConnection();

ClienteRepository clienteRepo = new ClienteRepository(database);
VeiculoRepository veiculoRepo = new VeiculoRepository(database);
MecanicoRepository mecanicoRepo = new MecanicoRepository(database);
ServicoRepository servicoRepo = new ServicoRepository(database);
PecaRepository pecaRepo = new PecaRepository(database);
OrdemServicoRepository ordemRepo = new OrdemServicoRepository(database);
OrdemMecanicoRepository ordemMecanicoRepo = new OrdemMecanicoRepository(database);
ItemServicoRepository itemServicoRepo = new ItemServicoRepository(database);
ItemPecaRepository itemPecaRepo = new ItemPecaRepository(database);
NotaFiscalRepository notaRepo = new NotaFiscalRepository(database);

System.out.println("Repositorios inicializados!");

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Opened database successfully
Repositorios inicializados!


## 2. Cliente chega na oficina
Um novo cliente chega na oficina com seu veículo para avaliação.

In [4]:
Cliente cliente = new Cliente();
cliente.setNome("Roberto Alves");
cliente.setCpfCnpj("444.444.444-44");
cliente.setEndereco("Av. Brasil");
cliente.setCep("74100-000");
cliente.setNumero("500");
cliente.setComplemento("");
clienteRepo.create(cliente);

Veiculo veiculo = new Veiculo();
veiculo.setPlaca("XYZ-9999");
veiculo.setModelo("HB20");
veiculo.setAnoModelo(2019);
veiculo.setCliente(cliente);
veiculoRepo.create(veiculo);

System.out.println("Cliente cadastrado: " + cliente.getNome() + " id=" + cliente.getId());
System.out.println("Veiculo cadastrado: " + veiculo.getModelo() + " placa=" + veiculo.getPlaca());

Cliente cadastrado: Roberto Alves id=20
Veiculo cadastrado: HB20 placa=XYZ-9999


## 3. Abertura da Ordem de Serviço
O mecânico avalia o veículo e abre uma ordem de serviço.

In [5]:
Mecanico mecanico = new Mecanico();
mecanico.setNome("Lucas Ferreira");
mecanicoRepo.create(mecanico);

OrdemServico ordem = new OrdemServico();
ordem.setDataEmissao(new Date());
ordem.setValorTotal(0.0);
ordem.setVeiculo(veiculo);
ordemRepo.create(ordem);

OrdemMecanico ordemMecanico = new OrdemMecanico();
ordemMecanico.setOrdemServico(ordem);
ordemMecanico.setMecanico(mecanico);
ordemMecanicoRepo.create(ordemMecanico);

System.out.println("Mecanico responsavel: " + mecanico.getNome());
System.out.println("Ordem de servico aberta: id=" + ordem.getId() + " data=" + ordem.getDataEmissao());

Mecanico responsavel: Lucas Ferreira
Ordem de servico aberta: id=17 data=Tue Sep 15 22:22:15 BRT 2026


## 4. Adição de Serviços e Peças
O mecânico identifica os serviços necessários e as peças a serem utilizadas.

In [6]:
Servico servico1 = new Servico();
servico1.setDescricao("Troca de oleo");
servico1.setValor(80.0);
servicoRepo.create(servico1);

Servico servico2 = new Servico();
servico2.setDescricao("Revisao de freios");
servico2.setValor(120.0);
servicoRepo.create(servico2);

ItemServico itemServico1 = new ItemServico();
itemServico1.setServico(servico1);
itemServico1.setOrdemServico(ordem);
itemServico1.setQuantidade(1);
itemServicoRepo.create(itemServico1);

ItemServico itemServico2 = new ItemServico();
itemServico2.setServico(servico2);
itemServico2.setOrdemServico(ordem);
itemServico2.setQuantidade(1);
itemServicoRepo.create(itemServico2);

System.out.println("Servico adicionado: " + servico1.getDescricao() + " R$" + servico1.getValor());
System.out.println("Servico adicionado: " + servico2.getDescricao() + " R$" + servico2.getValor());

Servico adicionado: Troca de oleo R$80.0
Servico adicionado: Revisao de freios R$120.0


In [7]:
Peca peca1 = new Peca();
peca1.setNome("Oleo de motor 5W30");
peca1.setPrecoUnitario(45.0);
pecaRepo.create(peca1);

Peca peca2 = new Peca();
peca2.setNome("Pastilha de freio dianteira");
peca2.setPrecoUnitario(90.0);
pecaRepo.create(peca2);

ItemPeca itemPeca1 = new ItemPeca();
itemPeca1.setPeca(peca1);
itemPeca1.setOrdemServico(ordem);
itemPeca1.setQuantidade(4);
itemPecaRepo.create(itemPeca1);

ItemPeca itemPeca2 = new ItemPeca();
itemPeca2.setPeca(peca2);
itemPeca2.setOrdemServico(ordem);
itemPeca2.setQuantidade(2);
itemPecaRepo.create(itemPeca2);

System.out.println("Peca adicionada: " + peca1.getNome() + " qtd=4 R$" + (peca1.getPrecoUnitario() * 4));
System.out.println("Peca adicionada: " + peca2.getNome() + " qtd=2 R$" + (peca2.getPrecoUnitario() * 2));

Peca adicionada: Oleo de motor 5W30 qtd=4 R$180.0
Peca adicionada: Pastilha de freio dianteira qtd=2 R$180.0


## 5. Finalização da Ordem
Cálculo do valor total e fechamento da ordem de serviço.

In [8]:
double totalServicos = itemServicoRepo.findAll().stream()
    .filter(is -> is.getOrdemServico().getId() == ordem.getId())
    .mapToDouble(is -> is.getServico().getValor() * is.getQuantidade())
    .sum();

double totalPecas = itemPecaRepo.findAll().stream()
    .filter(ip -> ip.getOrdemServico().getId() == ordem.getId())
    .mapToDouble(ip -> ip.getPeca().getPrecoUnitario() * ip.getQuantidade())
    .sum();

double valorTotal = totalServicos + totalPecas;
ordem.setValorTotal(valorTotal);
ordemRepo.update(ordem);

System.out.println("Total servicos: R$" + totalServicos);
System.out.println("Total pecas: R$" + totalPecas);
System.out.println("Valor total da ordem: R$" + valorTotal);

Total servicos: R$200.0
Total pecas: R$360.0
Valor total da ordem: R$560.0


## 6. Emissão da Nota Fiscal
Com a ordem finalizada, a nota fiscal é emitida para o cliente.

In [9]:
double valorImposto = valorTotal * 0.1;

NotaFiscal nota = new NotaFiscal();
nota.setNumero("NF-" + System.currentTimeMillis());
nota.setChaveAcesso("CHAVE" + String.format("%039d", ordem.getId()));
nota.setDataEmissao(new Date());
nota.setValorImposto(valorImposto);
nota.setOrdemServico(ordem);
notaRepo.create(nota);

System.out.println("Nota fiscal emitida: " + nota.getNumero());
System.out.println("Valor imposto (10%): R$" + valorImposto);
System.out.println("Valor total com imposto: R$" + (valorTotal + valorImposto));

Nota fiscal emitida: NF-1789521802188
Valor imposto (10%): R$56.0
Valor total com imposto: R$616.0


## 7. Resumo do Atendimento
Visão completa do atendimento realizado.

In [10]:
System.out.println("========================================");
System.out.println("RESUMO DO ATENDIMENTO");
System.out.println("========================================");
System.out.println("Cliente: " + cliente.getNome());
System.out.println("Veiculo: " + veiculo.getModelo() + " (" + veiculo.getPlaca() + ")");
System.out.println("Mecanico: " + mecanico.getNome());
System.out.println("----------------------------------------");
System.out.println("Servicos:");
itemServicoRepo.findAll().stream()
    .filter(is -> is.getOrdemServico().getId() == ordem.getId())
    .forEach(is -> System.out.println("  - " + is.getServico().getDescricao() + " x" + is.getQuantidade() + " R$" + (is.getServico().getValor() * is.getQuantidade())));
System.out.println("Pecas:");
itemPecaRepo.findAll().stream()
    .filter(ip -> ip.getOrdemServico().getId() == ordem.getId())
    .forEach(ip -> System.out.println("  - " + ip.getPeca().getNome() + " x" + ip.getQuantidade() + " R$" + (ip.getPeca().getPrecoUnitario() * ip.getQuantidade())));
System.out.println("----------------------------------------");
System.out.println("Valor total: R$" + ordem.getValorTotal());
System.out.println("Imposto (10%): R$" + nota.getValorImposto());
System.out.println("Nota fiscal: " + nota.getNumero());
System.out.println("========================================");

RESUMO DO ATENDIMENTO
Cliente: Roberto Alves
Veiculo: HB20 (XYZ-9999)
Mecanico: Lucas Ferreira
----------------------------------------
Servicos:
  - Troca de oleo x1 R$80.0
  - Revisao de freios x1 R$120.0
Pecas:
  - Oleo de motor 5W30 x4 R$180.0
  - Pastilha de freio dianteira x2 R$180.0
----------------------------------------
Valor total: R$560.0
Imposto (10%): R$56.0
Nota fiscal: NF-1789521802188


## 8. Encerramento
Fechamento da conexão com o banco.

In [11]:
database.close();
System.out.println("Conexao encerrada.");

Conexao encerrada.
